# 05 — Train · my pipeline


Trains on the dataset built by notebook 03: three-class molecular subtype,
I-SPY2 only, 80 mm physical crop, 8 spread slices.

**Where this stands.** Fifteen runs across four data configurations and two
architectures put this task between 0.55 and 0.63 macro-AUC. The ceiling does not
move with preprocessing, architecture, normalisation, augmentation or freezing.
That is a result, not a failure — and 0.58 carries enough signal to measure
federated degradation, which is what the thesis is about.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import dataset_config as config
from dataset_config import Config, TASKS

plt.rcParams.update({"figure.dpi": 120, "font.size": 9, "axes.grid": True,
                     "grid.alpha": 0.25, "axes.spines.top": False,
                     "axes.spines.right": False})

from core.training import run
from core.models import SUPPORTED

## Choose the model — one line

Every model below is implemented and tested. Uncomment exactly one.

In [ ]:
for name, note in SUPPORTED.items():
    print(f"# MODEL = {name!r:<20} # {note}")

In [ ]:
# ---- the three lines that define this experiment ---------------------------
MODEL = "resnet18"       # the measured winner; see the list above
SEED  = 42

from dataset_config import lr_for_model
CFG = Config(pipeline="mine", task="subtype", model=MODEL, seed=SEED,
             cohorts=("spy2",), learning_rate=lr_for_model(MODEL))

# Ablation switches — change one at a time, never two.
# CFG.freeze_until = "layer3"     # freeze conv1+bn1+layer1+layer2 (6% of params)
# CFG.augmentation = "half"       # measured: tripled the gap, lost 0.040 AUC
# CFG.mixup_alpha  = 0.2

## What is about to run

In [ ]:
print(CFG.summary())

## Train

One call. Everything is written automatically into an auto-numbered folder under
`results/`: curves, ROC, precision-recall, confusion matrix, per-class metrics,
the classification report, per-patient predictions and both checkpoints. Nothing
is left to do afterwards.

In [ ]:
run_dir = run(CFG)
print("\nfiles written:")
for f in sorted(run_dir.rglob("*")):
    if f.is_file():
        print("  ", f.relative_to(run_dir))

## The figures this run produced

In [ ]:
from IPython.display import Image as Show, display
for name in ["loss_curve.png", "accuracy_curve.png", "auc_and_gap_curve.png"]:
    p = run_dir / name
    if p.exists():
        display(Show(filename=str(p)))
for p in sorted((run_dir / "figures").glob("*_test.png")):
    display(Show(filename=str(p)))

## Reading the result

1. **Patient-level macro-AUC is the headline.** Slice-level numbers are reported
   only as an overfitting signal.
2. **Never quote accuracy without the trivial baseline** printed beside it.
3. **Treat any difference below 0.067 macro-AUC as noise.** Two byte-identical
   configurations differing only in seed were measured that far apart.
4. **One seed is not a result.** Repeat with `Config(seed=1)` before concluding.